### Python implementation of Leen's Network (minimal)

This notebook is an example of code implementation of Leen's algorithm to find principal components. This notebook focuses on the core algorithm of the Leen's minimal network, which is a simplified version of the full Leen's network. The minimal network is designed to capture the essential features of the algorithm while accurately capturing all the PCs of the input data. 

The math derivations are below:

$$
\mathbf{y} = \mathbf{w} \mathbf{x}
$$

$$
S = (I + q) y
$$

$$
\Delta W_i = \eta_w (\langle x s_i \rangle - \gamma_w \langle y_i^2 \rangle w_i)
$$

$$
\Delta q_{ik} = \eta_q (\langle y_i^2 + y_k^2 \rangle) (- q_{ij} - C\langle y_i y_j \rangle), q_{ii} = 0
$$



In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

In [2]:
class LeenMinimalPCA:
    """
    Activity-dependent Leen/Földiak PCA learner (NumPy).

    y = (I - q)^{-1} W x        # complete mode
    or y = W x                  # minimal mode (fast, robust for learning)

    Lateral (off-diagonal only):
        q_{ij} <- q_{ij} + η_q * (λ_i + λ_j) * ( - q_{ij} - C * <y_i y_j> )

    Forward (per unit i):
        w_i <- w_i + η_W * ( < x * s_i > - < y_i^2 > w_i )
        where s = y + q y  (vector per sample); i.e. s = (I + q) y

    Parameters
    ----------
    input_dim : int
    output_dim : int
    eta_w : float            # forward learning rate
    eta_q : float            # lateral learning rate (make this larger than eta_w)
    C : float                # coupling; pick > 1 (e.g., 1.5)
    ema_alpha : float        # EMA step for activity λ_i ≈ E[y_i^2]
    forward_mode : str       # "minimal", "complete", or "complete_first_order"
    symmetrize_q : bool      # keep q explicitly symmetric + zero diagonal after each step
    clip_q_spectral : float or None  # if not None, shrink q to keep ||q||_2 <= this value (e.g., 0.95)
    seed : int or None
    """
    def __init__(self,
                 input_dim: int,
                 output_dim: int,
                 eta_w: float = 1e-3,
                 eta_q: float = 1e-2,
                 C: float = 1.5,
                 ema_alpha: float = 0.05,
                 symmetrize_q: bool = True,
                 clip_q_spectral: float | None = None,
                 seed: int | None = 0):

        self.d = input_dim
        self.m = output_dim
        self.eta_w = eta_w
        self.eta_q = eta_q
        self.C = C
        self.ema_alpha = ema_alpha
        self.symmetrize_q = symmetrize_q
        self.clip_q_spectral = clip_q_spectral

        rng = np.random.default_rng(seed)
        # Row-wise unit-norm init for W
        W = rng.normal(size=(self.m, self.d))
        W /= np.linalg.norm(W, axis=1, keepdims=True) + 1e-12
        self.W = W
        # Symmetric zero-diagonal init for q
        self.q = np.zeros((self.m, self.m))
        np.fill_diagonal(self.q, 0.0)
        # EMA of activities (start small positive to avoid zero)
        self.lam = np.full(self.m, 1e-6)

    # ---------- helpers ----------

    def _forward(self, X: np.ndarray) -> np.ndarray:
        """
        X: (B, d) -> Y: (B, m)
        """
        Y = X @ self.W.T  # (B, m) = W x
        return Y

    def _symmetrize_q(self):
        self.q = 0.5 * (self.q + self.q.T)
        np.fill_diagonal(self.q, 0.0)

    def _clip_q_spectral_norm(self):
        if self.clip_q_spectral is None:
            return
        # Spectral norm via SVD
        u, s, vt = np.linalg.svd(self.q, full_matrices=False)
        smax = s[0]
        if smax > self.clip_q_spectral:
            s = s * (self.clip_q_spectral / (smax + 1e-12))
            self.q = (u * s) @ vt
            self._symmetrize_q()

    # ---------- public API ----------

    def step(self, X: np.ndarray):
        """
        One learning step on a batch X: shape (B, d).
        """
        X = np.asarray(X, dtype=float)
        B = X.shape[0]

        # Forward pass
        Y = self._forward(X)                                     # (B, m)

        # Batch moments
        y_cov = (Y.T @ Y) / B                                    # (m, m)  ~ < y_i y_j >
        y_var = np.diag(y_cov).copy()                            # (m,)

        # EMA update of activities λ_i ~ E[y_i^2]
        self.lam = (1.0 - self.ema_alpha) * self.lam + self.ema_alpha * y_var

        # ----- Lateral update (off-diagonal) -----
        lam_sum = self.lam[:, None] + self.lam[None, :]          # (m, m)  (λ_i + λ_j)
        dq = self.eta_q * lam_sum * ( - self.q - self.C * y_cov )
        np.fill_diagonal(dq, 0.0)
        self.q += dq

        if self.symmetrize_q:
            self._symmetrize_q()
        self._clip_q_spectral_norm()

        # ----- Forward update (Hebb-Oja, gated by q) -----
        # s = (I + q) y  (per sample); with row-vectors: S = Y @ (I+q)^T
        S = Y @ (np.eye(self.m) + self.q).T                      # (B, m)
        # < x * s_i > as a matrix: (d, m) then transpose to (m, d)
        XS = (X.T @ S) / B                                       # (d, m)
        dW = self.eta_w * (XS.T - y_var[:, None] * self.W)       # (m, d)
        self.W += dW

        # Mild row renorm to avoid drift (Oja already stabilizes norms)
        row_norms = np.linalg.norm(self.W, axis=1, keepdims=True) + 1e-12
        self.W /= row_norms

    def transform(self, X: np.ndarray) -> np.ndarray:
        "Project X onto learned components (uses the current forward mode)."
        return self._forward(np.asarray(X, dtype=float))

    @property
    def components_(self) -> np.ndarray:
        "Rows of W are the learned components (unit-norm). Shape: (m, d)"
        return self.W.copy()



In [4]:
class LeenCompletePCA:
    """
    Implementation of Leen's "complete" PCA model (Section 4).

    In this model, the neuron's output `y` is calculated recurrently,
    meaning the lateral connections `q` (eta in the paper) directly
    influence the cell's response in a feedback loop.

    Equations of Motion (ensemble average form):
    -------------------------------------------
    Forward Pass (Eq. 33):
        y = (I - q)^-1 W x

    Forward Weight Update (Eq. 35):
        Δw = η_w * ( <y x^T> - Diag(<y y^T>) w )

    Lateral Connection Update (Activity-Dependent, Eq. 47):
        Δq_ij = η_q * ( (<y_i^2> + <y_j^2>) * q_ij - C * <y_i y_j> )

    Parameters
    ----------
    input_dim : int
    output_dim : int
    eta_w : float             # Forward learning rate
    eta_q : float             # Lateral learning rate (make this larger than eta_w)
    C : float                 # Coupling; pick > 1 (e.g., 1.5)
    ema_alpha : float         # EMA step for activity λ_i ≈ E[y_i^2]
    symmetrize_q : bool       # Keep q symmetric + zero diagonal after each step
    seed : int or None
    """
    def __init__(self,
                 input_dim: int,
                 output_dim: int,
                 eta_w: float = 1e-3,
                 eta_q: float = 1e-2,
                 C: float = 1.5,
                 ema_alpha: float = 0.05,
                 symmetrize_q: bool = True,
                 seed: int | None = 0):
        self.d = input_dim
        self.m = output_dim
        self.eta_w = eta_w
        self.eta_q = eta_q
        self.C = C
        self.ema_alpha = ema_alpha
        self.symmetrize_q = symmetrize_q

        rng = np.random.default_rng(seed)
        # Row-wise unit-norm init for W
        W = rng.normal(size=(self.m, self.d))
        W /= np.linalg.norm(W, axis=1, keepdims=True) + 1e-12
        self.W = W
        # Symmetric zero-diagonal init for q
        self.q = np.zeros((self.m, self.m))
        # EMA of activities (start small positive to avoid zero)
        self.lam = np.full(self.m, 1e-6)

    # ---------- helpers ----------

    def _forward(self, X: np.ndarray) -> np.ndarray:
        """
        Calculates the recurrent cell response y = (I - q)^-1 W x
        X: (B, d) -> Y: (B, m)
        """
        Z = X @ self.W.T  # (B, m), represents Wx
        L = np.eye(self.m) - self.q
        try:
            # Solves the linear system (I - q)y = Wx for y
            Y = np.linalg.solve(L, Z.T).T
        except np.linalg.LinAlgError:
            print("Warning: Singular matrix in forward pass. Using pseudo-inverse.")
            L_inv = np.linalg.pinv(L)
            Y = Z @ L_inv.T
        return Y

    def _symmetrize_q(self):
        self.q = 0.5 * (self.q + self.q.T)
        np.fill_diagonal(self.q, 0.0)

    # ---------- public API ----------

    def step(self, X: np.ndarray):
        """
        One learning step on a batch X: shape (B, d).
        """
        X = np.asarray(X, dtype=float)
        B = X.shape[0]

        # Forward pass (recurrent calculation)
        Y = self._forward(X)  # (B, m)

        # Batch moments from the complete response
        y_cov = (Y.T @ Y) / B      # (m, m) ~ <y y^T>
        y_var = np.diag(y_cov).copy()  # (m,)   ~ <y_i^2>

        # EMA update of activities λ_i ~ E[y_i^2]
        self.lam = (1.0 - self.ema_alpha) * self.lam + self.ema_alpha * y_var

        # ----- Lateral update (Activity-Dependent Anti-Hebbian) -----
        lam_sum = self.lam[:, None] + self.lam[None, :]  # (m, m) (λ_i + λ_j)
        # Implements Δq_ij = η_q * ( (λ_i + λ_j) * q_ij - C * <y_i y_j> )
        dq = self.eta_q * (lam_sum * self.q - self.C * y_cov)
        np.fill_diagonal(dq, 0.0)
        self.q += dq

        if self.symmetrize_q:
            self._symmetrize_q()

        # ----- Forward update (Hebb-Oja on complete response) -----
        # Implements Δw = η_w * ( <y x^T> - Diag(<y y^T>) w )
        hebb_term = (Y.T @ X) / B  # (m, d)
        oja_term = y_var[:, None] * self.W      # (m, d)
        dW = self.eta_w * (hebb_term - oja_term)
        self.W += dW

    def transform(self, X: np.ndarray) -> np.ndarray:
        "Project X onto learned components using the recurrent dynamics."
        return self._forward(np.asarray(X, dtype=float))

    @property
    def components_(self) -> np.ndarray:
        "Rows of W are the learned components. Shape: (m, d)"
        # Normalize for stability and consistency, as Oja's rule keeps them near norm 1
        W_norm = self.W.copy()
        W_norm /= np.linalg.norm(W_norm, axis=1, keepdims=True) + 1e-12
        return W_norm

In [3]:
# --- helpers ---
def best_match_alignment(W_rows, PC_rows):
    """
    W_rows: (m, d) learned components, row-normalized
    PC_rows: (m, d) true PCs, row-normalized in descending variance order
    Returns:
      perm: indices of PCs assigned to each W row
      corrs: absolute correlations after optimal assignment
      C: full |cosine| matrix (m x m)
    """
    # normalize
    Wn = W_rows / (np.linalg.norm(W_rows, axis=1, keepdims=True) + 1e-12)
    PCn = PC_rows / (np.linalg.norm(PC_rows, axis=1, keepdims=True) + 1e-12)
    # cosine matrix
    C = np.abs(Wn @ PCn.T)

        # Hungarian solves a min-cost problem; convert to cost = 1 - C
    r, c = linear_sum_assignment(1.0 - C)
        # # greedy fallback
        # C_copy = C.copy()
        # m = C.shape[0]
        # r, c = [], []
        # for _ in range(m):
        #     i, j = np.unravel_index(np.argmax(C_copy), C_copy.shape)
        #     r.append(i); c.append(j)
        #     C_copy[i, :] = -np.inf
        #     C_copy[:, j] = -np.inf
        # r = np.array(r); c = np.array(c)

    return c, C[np.arange(C.shape[0]), c], C


def true_pcs_rows(X, m):
    """Return top-m PCs as ROWS (shape m x d)."""
    Xc = X - X.mean(axis=0, keepdims=True)
    U, S, VT = np.linalg.svd(Xc, full_matrices=False)
    return VT[:m, :]  # rows are PCs


def subspace_max_angle_deg(W_rows, PC_rows):
    """Largest principal angle between the two m-dim subspaces (in degrees)."""
    # Orthonormal bases for colspaces of W^T and PC^T
    Qw, _ = np.linalg.qr(W_rows.T)
    Qp, _ = np.linalg.qr(PC_rows.T)
    s = np.linalg.svd(Qw.T @ Qp, compute_uv=False)
    return float(np.degrees(np.arccos(np.clip(s.min(), -1.0, 1.0))))

In [5]:
# Synthetic anisotropic Gaussian (d=20), learn top m=5 PCs
rng = np.random.default_rng(0)
d, m = 10,5
eigvals = np.linspace(5.0, 1.0, d)
A = rng.normal(size=(d, d))
Sigma = A @ np.diag(eigvals) @ A.T
# Draw a batch
B = 512
X = rng.multivariate_normal(mean=np.zeros(d), cov=Sigma, size=B)

model = LeenMinimalPCA(d, m, eta_w=3e-3, eta_q=1.5e-1, C=1.8,
                       ema_alpha=0.1,
                       clip_q_spectral=0.9, seed=None)

# model = LeenCompletePCA(d, m, eta_w=2e-4, eta_q=5e-3, C=5.0,
#                        ema_alpha=0.05, symmetrize_q=True, seed=None)


In [6]:
# e.g., run another 600–1200 steps
for step in range(30000):
    Xb = rng.multivariate_normal(np.zeros(d), Sigma, size=4096)
    model.step(Xb)

    # cosine-monitor every 100 steps (optional)
    if step % 100 == 0:
        W = model.components_
        PC = true_pcs_rows(Xb, model.m)
        _, best_corrs, _ = best_match_alignment(W, PC)
        angle = subspace_max_angle_deg(W, PC)
        print(f"step {step:4d} | angle={angle:5.1f}° | best|corr|={np.round(best_corrs,3)}")
        # print true PCs

    # simple LR anneal halfway
    if step == 600:
        model.eta_w *= 0.3
        # model.eta_q *= 10

step    0 | angle= 77.4° | best|corr|=[0.238 0.273 0.41  0.617 0.783]
step  100 | angle=  3.5° | best|corr|=[0.979 0.985 0.871 0.978 0.997]
step  200 | angle=  1.7° | best|corr|=[0.997 0.994 0.972 0.855 0.995]
step  300 | angle=  1.3° | best|corr|=[0.997 1.    0.961 0.914 0.998]
step  400 | angle=  3.0° | best|corr|=[0.999 0.995 0.872 0.814 0.995]
step  500 | angle=  1.7° | best|corr|=[0.997 0.998 0.865 0.799 0.996]
step  600 | angle=  1.7° | best|corr|=[0.999 0.999 0.808 0.895 1.   ]
step  700 | angle=  1.7° | best|corr|=[0.999 0.995 0.925 0.94  0.995]
step  800 | angle=  3.4° | best|corr|=[0.998 0.983 0.86  0.875 0.982]
step  900 | angle=  2.5° | best|corr|=[0.996 0.993 0.941 0.901 0.993]
step 1000 | angle=  1.8° | best|corr|=[0.999 0.982 0.834 0.844 0.983]
step 1100 | angle=  1.2° | best|corr|=[0.999 0.999 0.955 0.965 0.999]
step 1200 | angle=  1.6° | best|corr|=[0.999 0.998 0.991 0.995 0.999]
step 1300 | angle=  1.8° | best|corr|=[0.999 0.999 0.959 0.97  0.999]
step 1400 | angle=  

KeyboardInterrupt: 